# 2.3 Προετοιμασία δεδομένων (FLORES)

Φόρτωση του openlanguagedata/flores_plus

https://huggingface.co/datasets/openlanguagedata/flores_plus

In [ ]:
!pip install -q "transformers<5" datasets sentencepiece sacrebleu sacremoses

In [ ]:
import gc
import torch
from datasets import load_dataset
from huggingface_hub import notebook_login
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, MarianMTModel, MarianTokenizer

In [ ]:
# Πρόσβαση στο dataset
notebook_login()

In [ ]:
# Καλούμε και αποθηκεύουμε το ελληνικό και το αγγλικό devtest.
ell_dataset = load_dataset("openlanguagedata/flores_plus", "ell_Grek", split="devtest")
eng_dataset = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest")

## Διερεύνηση του flores

In [ ]:
print("Lengths:", len(eng_dataset), len(ell_dataset))
print("Types:", type(eng_dataset), type(ell_dataset))
print(f"Column_named_eng: {eng_dataset.column_names}")
print(f"Column_named_ell: {ell_dataset.column_names}")

In [ ]:
print(eng_dataset[0])
print(ell_dataset[0])
print("\n--- First 5 EN ---")
print(*eng_dataset["text"][:5], sep="\n")
print("\n--- First 5 EL ---")
print(*ell_dataset["text"][:5], sep="\n")

## Αποθήκευση των προτάσεων των dataset σε αρχεία txt

In [ ]:
with open("Flores_ell.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(ell_dataset["text"]) + "\n")

with open("Flores_eng.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(eng_dataset["text"]) + "\n")

# 2.4 Υλοποίηση μεταφράσεων

## Φόρτωση των δεδομένων

In [ ]:
with open("Flores_ell.txt", "r", encoding="utf-8") as f:
    ell_dataset = f.read().splitlines()

with open("Flores_eng.txt", "r", encoding="utf-8") as f:
    eng_dataset = f.read().splitlines()

with open("data/test.el", "r", encoding="utf-8") as f:
    test_ell = f.read().splitlines()

with open("data/test.en", "r", encoding="utf-8") as f:
    test_eng = f.read().splitlines()

print("FLORES:", len(ell_dataset), len(eng_dataset))
print("Test set:", len(test_ell), len(test_eng))

## 2.4.1 facebook/nllb-200-distilled-600M

In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="ell_Grek")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

### Δοκιμή

(https://huggingface.co/docs/transformers/model_doc/nllb)

In [ ]:
text = "Καλημέρα, σήμερα είναι Σάββατο και ο καιρός είναι ζεστός."
inputs = tokenizer(text, return_tensors="pt").to(model.device)

translated_tokens = model.generate(**inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"), max_length=30)
tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

### Εκτέλεση στο Flores Dataset

In [ ]:
batch_size = 32
model_translations = []

for i in range(0, len(ell_dataset), batch_size):
    batch = ell_dataset[i : i + batch_size]
    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(model.device)
    all_translations = model.generate(**inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"), max_length=256)
    batch_preds = tokenizer.batch_decode(all_translations, skip_special_tokens=True)
    model_translations.extend(batch_preds)

print("FLORES translations:", len(model_translations))

with open("facebook_nllb_200_translations.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(model_translations) + "\n")

### Εκτέλεση στο Test Set

In [ ]:
test_translations = []

for i in range(0, len(test_ell), batch_size):
    batch = test_ell[i : i + batch_size]
    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(model.device)
    all_translations = model.generate(**inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"), max_length=256)
    batch_preds = tokenizer.batch_decode(all_translations, skip_special_tokens=True)
    test_translations.extend(batch_preds)

print("Test set translations:", len(test_translations))

with open("facebook_nllb_200_test_translations.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(test_translations) + "\n")

del model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
elif torch.backends.mps.is_available():
    torch.mps.empty_cache()

## 2.4.2 Helsinki-NLP/opus-mt-grk-en

https://huggingface.co/Helsinki-NLP/opus-mt-grk-en

In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

model_name = "Helsinki-NLP/opus-mt-grk-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)

### Δοκιμή

In [ ]:
text = "Καλημέρα, σήμερα είναι Σάββατο και ο καιρός είναι ζεστός."
inputs = tokenizer(text, return_tensors="pt", padding=True).to(device)
translated = model.generate(**inputs)

print(tokenizer.batch_decode(translated, skip_special_tokens=True)[0])

### Εκτέλεση στο Flores Dataset

In [ ]:
batch_size = 32
model_translations = []

for i in range(0, len(ell_dataset), batch_size):
    batch = ell_dataset[i : i + batch_size]
    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
    translations = model.generate(**inputs, max_length=256)
    batch_preds = tokenizer.batch_decode(translations, skip_special_tokens=True)
    model_translations.extend(batch_preds)

print("FLORES translations:", len(model_translations))

with open("Helsinki_NLP_Translations.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(model_translations) + "\n")

### Εκτέλεση στο Test Set

In [ ]:
test_translations = []

for i in range(0, len(test_ell), batch_size):
    batch = test_ell[i : i + batch_size]
    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
    translations = model.generate(**inputs, max_length=256)
    batch_preds = tokenizer.batch_decode(translations, skip_special_tokens=True)
    test_translations.extend(batch_preds)

print("Test set translations:", len(test_translations))

with open("Helsinki_NLP_Test_Translations.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(test_translations) + "\n")

del model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
elif torch.backends.mps.is_available():
    torch.mps.empty_cache()